# LLM-Verified Concept Mapping

This notebook demonstrates how to add **LLM verification** to the Portiere concept mapping
pipeline. Instead of relying solely on embedding similarity and reranking, borderline mappings
are routed to an LLM for verification -- improving accuracy where it matters most, without
incurring LLM costs for every single mapping.

## Confidence Routing and LLM Verification

Portiere uses a confidence routing system to decide how each mapping is handled:

| Confidence Range | Status | Action | LLM Involved? |
|-----------------|--------|--------|---------------|
| >= 0.95 | AUTO | Auto-accepted, high confidence | No |
| 0.80 - 0.95 | VERIFY | LLM verifies the mapping | Yes |
| 0.70 - 0.80 | REVIEW | LLM reviews, may suggest alternatives | Yes |
| < 0.70 | MANUAL | Requires human review | Optional |

**Why not send everything to the LLM?** Cost and latency. High-confidence mappings (>= 0.95)
are already accurate -- sending them to an LLM wastes money and time. Only borderline mappings
(0.70 - 0.95) benefit from LLM verification. This keeps costs manageable while improving
accuracy where it matters most.

The LLM can:
- **Confirm** a borderline mapping (VERIFY -> accepted)
- **Suggest a better concept** for a mapping
- **Flag incorrect mappings** for human review

## What We Will Do

1. Configure Portiere with an LLM provider (OpenAI or Ollama)
2. Add data sources (patients and diagnoses)
3. Map schema and approve
4. Map concepts with LLM verification enabled
5. Inspect results: AUTO vs VERIFIED vs REVIEW
6. Compare with a no-LLM baseline
7. Analyze cost implications
8. Run ETL and validate

## Prerequisites

- Python 3.10+
- An OpenAI API key (or Ollama installed locally for free local inference)
- Sample data files in `example_assets/01.4_llm_verified_mapping/`

## Step 1: Install Dependencies

Install the Portiere SDK and the OpenAI client library. If you prefer a local LLM,
install Ollama instead.

In [1]:
!pip install portiere-health openai
# Or for local LLM: pip install portiere-health ollama


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 1b: Shared Athena Vocabulary Setup

Build (or reuse) BM25s and FAISS indexes from the Athena vocabulary download.
On the first run this takes ~10-30 minutes for FAISS embedding; subsequent runs
reuse the cached indexes.

In [2]:
from pathlib import Path
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


## Step 2: Set API Key

Set your OpenAI API key. In production, use environment variables or a secrets manager
rather than hardcoding keys.

In [3]:
import os

# Option A: Set directly (not recommended for production)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Option B: From environment (recommended)
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"

AssertionError: Set OPENAI_API_KEY environment variable

## Step 3: Import and Configure

We configure Portiere with:

- `knowledge_layer` with `backend="bm25s"` -- BM25s for terminology retrieval
- `embedding` -- SapBERT for biomedical semantic similarity
- `reranker` -- cross-encoder for candidate re-scoring
- `llm` -- LLM provider for borderline mapping verification

Portiere infers **local mode** when `knowledge_layer` is configured and no `api_key`
is provided. There is no need to set `mode` or `pipeline` explicitly -- the SDK
detects the right operating mode from your intent.

The `temperature=0.0` setting ensures deterministic, reproducible results -- critical for
medical terminology mapping where consistency matters.

In [4]:
from pathlib import Path

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

In [5]:
import os
import portiere
from portiere import PortiereConfig, KnowledgeLayerConfig
from portiere.config import EmbeddingConfig, RerankerConfig, LLMConfig

# For OpenAI:
config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",                   # BM25s for terminology retrieval
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
    embedding=EmbeddingConfig(
        provider="huggingface",
        model="cambridgeltl/SapBERT-from-PubMedBERT-fulltext",  # Biomedical embeddings
    ),
    reranker=RerankerConfig(
        provider="huggingface",
        model="cross-encoder/ms-marco-MiniLM-L-6-v2",          # Cross-encoder reranker
    ),
    llm=LLMConfig(
        provider="ollama",                    # Or "ollama" for local LLM
        # api_key=os.getenv("OPENAI_API_KEY"),  # Set your API key
        model="gpt-4o",                       # Or "llama3:70b" for Ollama
        temperature=0.0,                      # Deterministic for medical mapping
    ),
)

# Alternative: Local LLM via Ollama (free, no API key needed)
# config = PortiereConfig(
#     knowledge_layer=KnowledgeLayerConfig(
#         backend="bm25s",
#         bm25s_corpus_path=str(BM25S_CORPUS),
#     ),
#     embedding=EmbeddingConfig(
#         provider="huggingface",
#         model="cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
#     ),
#     reranker=RerankerConfig(
#         provider="huggingface",
#         model="cross-encoder/ms-marco-MiniLM-L-6-v2",
#     ),
#     llm=LLMConfig(
#         provider="ollama",
#         endpoint="http://localhost:11434",
#         model="llama3:70b",
#         temperature=0.0,
#     ),
# )

print(f"Mode: {config.effective_mode}")
print(f"Pipeline: {config.effective_pipeline}")
print(f"Knowledge backend: {config.knowledge_layer.backend}")
print(f"LLM provider: {config.llm.provider}")
print(f"LLM model: {config.llm.model}")

Mode: local
Pipeline: local
Knowledge backend: bm25s
LLM provider: ollama
LLM model: gpt-4o


## Step 4: Create a Project

Initialize a new mapping project targeting OMOP CDM v5.4. The project uses standard
vocabularies (SNOMED, LOINC, RxNorm, ICD10CM) for concept mapping.

In [6]:
from portiere.engines import PolarsEngine

project = portiere.init(
    name="LLM-Verified Mapping Demo",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config,
)

print(project)

2026-04-16 00:31:39 [info     ] PolarsEngine initialized      


2026-04-16 00:31:39 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project(name='LLM-Verified Mapping Demo', task='standardize', target_model='omop_cdm_v5.4', mode='local')


## Step 5: Add Data Sources

We add two data sources:

- **patients.csv** -- 15 patients with 11 demographic columns (for schema mapping)
- **diagnoses.csv** -- 20 diagnosis records with ICD-10 codes (for concept mapping)

The diagnoses file is where LLM verification will shine -- it contains clinical codes
that need to be mapped to standard OMOP concepts.

In [7]:
patients_source = project.add_source(
    "example_assets/01.4_llm_verified_mapping/patients.csv"
)

print(f"Source added: {patients_source['name']}, format: {patients_source['format']}")
print(f"Columns detected: {patients_source.get('columns', 'N/A')}")
print(f"Row count: {patients_source.get('row_count', 'N/A')}")

2026-04-16 00:31:39 [info     ] project.source_added           project='LLM-Verified Mapping Demo' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:31:41 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


2026-04-16 00:31:41 [info     ] project.profiled               source=patients


Source added: patients, format: csv
Columns detected: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'address', 'city', 'state', 'zip_code']
Row count: 15


In [8]:
diagnoses_source = project.add_source(
    "example_assets/01.4_llm_verified_mapping/diagnoses.csv"
)

print(f"Source added: {diagnoses_source['name']}, format: {diagnoses_source['format']}")
print(f"Columns detected: {diagnoses_source.get('columns', 'N/A')}")
print(f"Row count: {diagnoses_source.get('row_count', 'N/A')}")

2026-04-16 00:31:41 [info     ] project.source_added           project='LLM-Verified Mapping Demo' source=diagnoses


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:31:41 [info     ] gx_profiler.profiled           columns=6 rows=20 source=diagnoses


2026-04-16 00:31:41 [info     ] project.profiled               source=diagnoses


Source added: diagnoses, format: csv
Columns detected: ['encounter_id', 'patient_id', 'diagnosis_code', 'diagnosis_description', 'diagnosis_type', 'diagnosis_date']
Row count: 20


## Step 6: Map Schema

Schema mapping automatically matches source columns to OMOP CDM target tables and columns.
The SDK uses the knowledge layer to find the best matches and assigns confidence scores.

In [9]:
schema_map = project.map_schema(patients_source)

print(f"Summary: {schema_map.summary()}")
print("\nSchema Mapping Results:")
print("-" * 80)

for item in schema_map.items:
    print(
        f"  {item.source_column} -> {item.target_table}.{item.target_column} "
        f"(confidence: {item.confidence:.2f}, status: {item.status.value})"
    )

2026-04-16 00:31:41 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:31:42 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:31:42 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:31:42 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:31:43 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:31:43 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:31:43 [info     ] local_storage.schema_mapping_saved items_count=11 project='LLM-Verified Mapping Demo'


2026-04-16 00:31:43 [info     ] project.schema_mapped          auto=10 project='LLM-Verified Mapping Demo' total=11


Summary: {'total': 11, 'auto_accepted': 10, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 90.9090909090909}

Schema Mapping Results:
--------------------------------------------------------------------------------
  patient_id -> person.person_id (confidence: 0.95, status: auto_accepted)
  first_name -> person.person_source_value (confidence: 0.95, status: auto_accepted)
  last_name -> person.person_source_value (confidence: 0.50, status: unmapped)
  date_of_birth -> person.birth_datetime (confidence: 0.95, status: auto_accepted)
  gender -> person.gender_concept_id (confidence: 0.95, status: auto_accepted)
  race -> person.race_concept_id (confidence: 0.95, status: auto_accepted)
  ethnicity -> person.ethnicity_concept_id (confidence: 0.95, status: auto_accepted)
  address -> location.address_1 (confidence: 0.95, status: auto_accepted)
  city -> location.city (confidence: 0.95, status: auto_accepted)
  state -> location.state (confidence: 0.95, status: auto_accepted)
 

## Step 7: Approve Schema

Approve all schema mappings and finalize. In a production workflow, you would selectively
review and override individual items before calling `finalize()`.

In [10]:
schema_map.approve_all()
schema_map.finalize()

print("Schema mapping finalized.")
print(f"Updated summary: {schema_map.summary()}")

Schema mapping finalized.
Updated summary: {'total': 11, 'auto_accepted': 10, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 90.9090909090909}


## Step 8: Map Concepts with LLM Verification

This is the key step. When `map_concepts()` runs with an LLM configured, the pipeline:

1. Retrieves candidate concepts using BM25s + embeddings
2. Re-ranks candidates with the cross-encoder
3. Assigns confidence scores based on the reranked results
4. Routes borderline mappings (0.70 - 0.95) to the LLM for verification
5. Auto-accepts high-confidence mappings (>= 0.95) without LLM cost

The diagnoses source contains ICD-10 codes that will be mapped to standard OMOP concepts
(primarily SNOMED).

In [11]:
concept_map = project.map_concepts(source=diagnoses_source)

print(f"Summary: {concept_map.summary()}")

2026-04-16 00:31:43 [info     ] project.code_columns_detected  columns=['diagnosis_code']


2026-04-16 00:31:43 [info     ] Stage 3: Concept mapping (local) columns=['diagnosis_code'] knowledge_backend=bm25s


2026-04-16 00:31:43 [info     ] Mapping column: diagnosis_code


2026-04-16 00:31:43 [info     ] Found description column: diagnosis_description code_column=diagnosis_code desc_column=diagnosis_description mappings=8


2026-04-16 00:31:43 [info     ] knowledge.creating_backend     backend=bm25s path=example_assets/vocabulary_download_v5/indexes/concepts.json


...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-16 00:31:52 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=8


2026-04-16 00:31:53 [info     ] local_storage.concept_mapping_saved items_count=8 project='LLM-Verified Mapping Demo'


2026-04-16 00:31:53 [info     ] project.concepts_mapped        auto_rate=100.0 project='LLM-Verified Mapping Demo' total=8


Summary: {'total': 8, 'auto_mapped': 8, 'needs_review': 0, 'manual_required': 0, 'auto_rate': 100.0, 'coverage': 100.0}


## Step 9: Inspect Results -- AUTO vs VERIFIED vs REVIEW

Each concept mapping item has a `method` field indicating how the mapping was determined:

- `auto` -- High confidence (>= 0.95), accepted without LLM
- `verified` -- LLM confirmed the mapping (was 0.80 - 0.95)
- `review` -- LLM reviewed but could not fully confirm (0.70 - 0.80)
- `manual` -- Needs human review (< 0.70)

Let's see the distribution of methods and inspect individual results.

In [12]:
from collections import Counter

method_counts = Counter(item.method.value for item in concept_map.items)
print("Mapping methods:")
for method, count in method_counts.items():
    print(f"  {method}: {count}")

print("\nDetailed results:")
print("-" * 80)
for item in concept_map.items:
    print(
        f"  {item.source_code}: {item.target_concept_name} "
        f"(method: {item.method.value}, confidence: {item.confidence:.2f})"
    )

Mapping methods:
  auto: 8

Detailed results:
--------------------------------------------------------------------------------
  I10: Essential hypertension (method: auto, confidence: 8.70)
  R51: Headache (method: auto, confidence: 5.01)
  E11.9: Lipoatrophic diabetes mellitus without complication (method: auto, confidence: 10.36)
  N18.3: Chronic kidney disease stage 3 (method: auto, confidence: 9.97)
  F32.9: Major depression, single episode (method: auto, confidence: 12.28)
  J06.9: Acute upper respiratory infection (method: auto, confidence: 9.53)
  J44.1: Acute exacerbation of chronic obstructive pulmonary disease (method: auto, confidence: 13.55)
  Z87.891: Nicotine dependence (method: auto, confidence: 8.21)


### Items That Need Review

Check which items still need human attention after LLM verification.

In [13]:
review_concepts = concept_map.needs_review()

print(f"Concept mappings needing review: {len(review_concepts)}")
for item in review_concepts:
    print(
        f"  {item.source_code}: {item.target_concept_name} "
        f"(confidence: {item.confidence:.2f})"
    )

Concept mappings needing review: 0


## Step 10: Before/After Comparison

To demonstrate the improvement that LLM verification provides, let's run the same pipeline
**without** an LLM and compare the results. Without LLM verification, borderline mappings
remain in VERIFY or REVIEW status instead of being confirmed or corrected.

In [14]:
# Same pipeline without LLM for comparison
config_no_llm = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
    reranker=RerankerConfig(provider="none"),
    # No llm config -- defaults to provider="none"
)

project_no_llm = portiere.init(
    name="No-LLM Comparison",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config_no_llm,
)

# Add the same diagnoses source
diagnoses_no_llm = project_no_llm.add_source(
    "example_assets/01.4_llm_verified_mapping/diagnoses.csv"
)

# Map concepts without LLM
concept_map_no_llm = project_no_llm.map_concepts(source=diagnoses_no_llm)

print("Without LLM:")
print(f"  Summary: {concept_map_no_llm.summary()}")

method_counts_no_llm = Counter(item.method.value for item in concept_map_no_llm.items)
print(f"  Methods: {dict(method_counts_no_llm)}")

print("\nWith LLM:")
print(f"  Summary: {concept_map.summary()}")
print(f"  Methods: {dict(method_counts)}")

2026-04-16 00:31:53 [info     ] PolarsEngine initialized      


2026-04-16 00:31:53 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


2026-04-16 00:31:53 [info     ] project.source_added           project='No-LLM Comparison' source=diagnoses


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:31:53 [info     ] gx_profiler.profiled           columns=6 rows=20 source=diagnoses


...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-16 00:32:01 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=8


2026-04-16 00:32:01 [info     ] local_storage.concept_mapping_saved items_count=8 project='No-LLM Comparison'


2026-04-16 00:32:01 [info     ] project.concepts_mapped        auto_rate=100.0 project='No-LLM Comparison' total=8


Without LLM:
  Summary: {'total': 8, 'auto_mapped': 8, 'needs_review': 0, 'manual_required': 0, 'auto_rate': 100.0, 'coverage': 100.0}
  Methods: {'auto': 8}

With LLM:
  Summary: {'total': 8, 'auto_mapped': 8, 'needs_review': 0, 'manual_required': 0, 'auto_rate': 100.0, 'coverage': 100.0}
  Methods: {'auto': 8}


## Step 11: Cost Considerations

One of the key advantages of confidence routing is cost efficiency. Not every mapping
invokes the LLM -- only those that fall in the borderline confidence range (0.70 - 0.95).

With our 20 diagnosis records:
- High-confidence mappings (>= 0.95) are **auto-accepted** -- zero LLM cost
- Borderline mappings (0.70 - 0.95) are **LLM-verified** -- one LLM call each
- Low-confidence mappings (< 0.70) are flagged for **manual review**

Approximate costs per LLM call with GPT-4o: ~$0.01 - $0.03 USD.

For a completely free alternative, use **Ollama** with a local model like Llama 3.

In [15]:
llm_calls = sum(
    1 for item in concept_map.items
    if item.method.value in ("verified", "review")
)
auto_calls = sum(
    1 for item in concept_map.items
    if item.method.value == "auto"
)
manual_calls = sum(
    1 for item in concept_map.items
    if item.method.value == "manual"
)

print(f"Total concept mappings: {len(concept_map.items)}")
print(f"Auto-accepted (no LLM cost): {auto_calls}")
print(f"LLM-verified/reviewed: {llm_calls}")
print(f"Manual (human review): {manual_calls}")
print(f"Estimated cost: ~${llm_calls * 0.02:.2f} (GPT-4o)")

Total concept mappings: 8
Auto-accepted (no LLM cost): 8
LLM-verified/reviewed: 0
Manual (human review): 0
Estimated cost: ~$0.00 (GPT-4o)


## Step 12: Alternative -- Ollama Configuration

If you want **100% local, free** LLM verification, use Ollama. It runs open-source models
on your own machine with no API keys or external calls.

### Setup

1. Install Ollama: https://ollama.ai
2. Pull a model: `ollama pull llama3:70b`
3. Use the config below:

In [16]:
# Alternative: 100% local with Ollama
# 1. Install Ollama: https://ollama.ai
# 2. Pull a model: ollama pull llama3:70b
# 3. Use this config:

ollama_config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
    embedding=EmbeddingConfig(
        provider="huggingface",
        model="cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
    ),
    reranker=RerankerConfig(
        provider="huggingface",
        model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    ),
    llm=LLMConfig(
        provider="ollama",
        endpoint="http://localhost:11434",
        model="llama3:70b",
        temperature=0.0,
    ),
)

print(f"Ollama config ready: {ollama_config.llm.provider} / {ollama_config.llm.model}")
print("Note: Ollama must be running locally to use this config.")

Ollama config ready: ollama / llama3:70b
Note: Ollama must be running locally to use this config.


## Step 13: Approve and Finalize Concept Mappings

After LLM verification, approve all concept mappings and finalize them for ETL.

In [17]:
concept_map.approve_all()

print("Concept mappings approved.")
print(f"Updated summary: {concept_map.summary()}")

Concept mappings approved.
Updated summary: {'total': 8, 'auto_mapped': 8, 'needs_review': 0, 'manual_required': 0, 'auto_rate': 100.0, 'coverage': 100.0}


## Step 14: Run ETL and Validate

Generate the transformed output files and run validation checks to ensure
target model conformance.

In [18]:
etl_result = project.run_etl(
    patients_source,
    output_dir="./output",
    schema_mapping=schema_map,
    concept_mapping=concept_map,
)

print(f"ETL complete. Output: {etl_result}")

2026-04-16 00:32:01 [info     ] Reading source                 format=csv path=example_assets/01.4_llm_verified_mapping/patients.csv


2026-04-16 00:32:01 [info     ] Processing table               columns=6 table=person


2026-04-16 00:32:01 [info     ] Processing table               columns=4 table=location


2026-04-16 00:32:01 [info     ] ETL complete                   duration=0.004134 success=True


2026-04-16 00:32:01 [info     ] project.etl_complete           output_dir=./output project='LLM-Verified Mapping Demo'


ETL complete. Output: success=True started_at=datetime.datetime(2026, 4, 15, 17, 32, 1, 872022, tzinfo=datetime.timezone.utc) completed_at=datetime.datetime(2026, 4, 15, 17, 32, 1, 876156, tzinfo=datetime.timezone.utc) duration_seconds=0.004134 source_path='example_assets/01.4_llm_verified_mapping/patients.csv' source_rows_read=15 output_path='./output' tables=[TableResult(table_name='person', rows_written=15, columns=['person_id', 'person_source_value', 'birth_datetime', 'gender_concept_id', 'race_concept_id', 'ethnicity_concept_id'], output_path='./output/person.csv', concept_columns_mapped=[], unmapped_concept_count=0), TableResult(table_name='location', rows_written=15, columns=['address_1', 'city', 'state', 'zip'], output_path='./output/location.csv', concept_columns_mapped=[], unmapped_concept_count=0)] total_rows_written=30 schema_mappings_applied=10 concept_mappings_applied=0 unmapped_columns=['last_name'] engine_name='polars' errors=[] warnings=[]


In [19]:
validation = project.validate(etl_result=etl_result)

print(f"Validation result: {validation}")

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:32:01 [info     ] gx_validator.validated         completeness=0.44 conformance=1.00 passed=False plausibility=0.44 table=location


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

2026-04-16 00:32:02 [info     ] gx_validator.validated         completeness=0.67 conformance=1.00 passed=False plausibility=0.70 table=person


2026-04-16 00:32:02 [info     ] project.validated              all_passed=False project='LLM-Verified Mapping Demo' tables=2


Validation result: {'tables': [{'table_name': 'location', 'passed': False, 'completeness_score': 0.4444444444444444, 'conformance_score': 1.0, 'plausibility_score': 0.4444444444444444, 'gx_result': {'success': False, 'results': [{'success': False, 'expectation_config': {'type': 'expect_column_to_exist', 'kwargs': {'batch_id': 'validate_location-location', 'column': 'location_id'}, 'meta': {}, 'id': '191fee04-56a2-412f-976e-8d50163f8a73', 'severity': 'critical'}, 'result': {}, 'meta': {}, 'exception_info': {'raised_exception': False, 'exception_traceback': None, 'exception_message': None}}, {'success': True, 'expectation_config': {'type': 'expect_column_to_exist', 'kwargs': {'batch_id': 'validate_location-location', 'column': 'address_1'}, 'meta': {}, 'id': '9a560d2a-c65e-47c0-9952-2a350707a3a3', 'severity': 'critical'}, 'result': {}, 'meta': {}, 'exception_info': {'raised_exception': False, 'exception_traceback': None, 'exception_message': None}}, {'success': False, 'expectation_config

---

## Summary

In this notebook we demonstrated LLM-verified concept mapping with Portiere:

1. **Configured** the SDK with embedding, reranking, and LLM verification layers
2. **Added** patient and diagnosis data sources
3. **Mapped schema** and approved the results
4. **Mapped concepts** with LLM verification -- borderline mappings were routed to GPT-4o
5. **Inspected results** -- saw the distribution of AUTO, VERIFIED, REVIEW, and MANUAL methods
6. **Compared** with a no-LLM baseline to quantify the improvement
7. **Analyzed costs** -- only borderline mappings invoked the LLM
8. **Ran ETL** and validated the output

### Key Takeaways

- LLM verification improves accuracy for borderline mappings (0.70 - 0.95 confidence)
- High-confidence mappings (>= 0.95) are auto-accepted without LLM cost
- Cost-efficient: only borderline mappings incur LLM charges
- Supports multiple LLM providers: OpenAI, Anthropic, Azure, Bedrock, Ollama
- Ollama provides a free, fully local alternative for air-gapped environments

### When to Use LLM Verification

| Scenario | Recommendation |
|----------|---------------|
| Quick prototyping | Skip LLM (`provider="none"`) |
| Production with budget | OpenAI GPT-4o |
| Air-gapped / on-prem | Ollama with Llama 3 |
| Maximum accuracy | Hybrid + Reranker + LLM |

### Next Steps

- **Cloud pipeline**: See `02_cloud_pipeline.ipynb` to use Portiere's cloud AI inference
  while keeping your data local.
- **BYO-LLM**: See `03_byo_llm_providers.ipynb` to explore all supported LLM providers
  (OpenAI, Anthropic, Ollama, Azure, AWS Bedrock) with advanced configuration options.